# 🚀 Noah's Ark OS — VSCode Setup Guide
---

## Prerequisites

### 1. Install VSCode Extensions
Open VSCode → Extensions panel (`Ctrl+Shift+X`) and install:
- **Python** (Microsoft)
- **Jupyter** (Microsoft)
- **Pylance** (Microsoft) — recommended

### 2. Install Python Dependencies
Open a terminal (`Ctrl+\`` `) and run:
```bash
pip install google-genai ipywidgets python-dotenv notebook
```
Then enable the ipywidgets extension for Jupyter:
```bash
jupyter nbextension enable --py widgetsnbextension --sys-prefix
```

### 3. Create Your `.env` File
In the **same folder** as this notebook, create a file named `.env`:
```
GEMINI_API_KEY=your_actual_api_key_here
```
> ⚠️ Never commit `.env` to Git. Add it to `.gitignore`.

### 4. Create a `.gitignore` (if using Git)
```
.env
*.json
__pycache__/
```

### 5. Select the Jupyter Kernel in VSCode
- Open this `.ipynb` file in VSCode
- Top-right corner → click **"Select Kernel"**
- Choose **"Python Environments..."** → select your Python interpreter
- If you don't see one, choose **"Install suggested extensions"**

### 6. Run the Notebook
Run cells **top to bottom** in order using `Shift+Enter` or the ▶ Run All button.

---

## Switching FROM Google Colab TO VSCode

| Step | Colab | VSCode |
|------|-------|--------|
| Secrets | Stored in Colab's secret panel | Stored in local `.env` file |
| Kernel | Colab's cloud Python | Your local Python environment |
| Files | `/content/` directory | Same folder as this notebook |
| Widgets | Works in Colab | Works in VSCode (same ipywidgets) |
| Display | `IPython.display` | Same — no change needed |

---
> **Chapter:** 2.1 | **Version:** v1.5-genai | **Status:** Active Development


In [1]:
# ==============================================================================================================================
# TILE T000: BIOS — VSCode / google-genai Edition
# ==============================================================================================================================
# VERSION: 4.0.0 | STATUS: Evolved
# PREDECESSORS: None (Entry Point)
# INPUT: EnvLoader instance (T118) carrying GEMINI_API_KEY and optional MODEL_NAME
# PROCESSING: Validates API key, creates google-genai Client, discovers models, locks model config
# OUTPUT: (provider_str, model_name_str, client_obj) — all three consumed by Master Assembly
# ------------------------------------------------------------------------------------------------------------------------------
# VERSION CHANGE (3.0 → 4.0):
#   - Migrated from deprecated 'google.generativeai' to 'google-genai' (google.genai)
#   - genai.configure() + genai.list_models() replaced by Client() + client.models.list()
#   - Now returns client object so T115 can reuse it (no double-initialisation)
#   - No interactive input() — safe for VSCode Jupyter kernel
# INSTALL: pip install google-genai
# ------------------------------------------------------------------------------------------------------------------------------

def boot_sequence(env_loader):
    """
    VSCode BIOS: Validates environment, creates genai Client,
    discovers available Gemini models, returns (provider, model_name, client).
    """
    from google import genai

    print("--- NOAH'S ARK OS v5.5: google-genai BOOT ---")

    # ── CONFIG BLOCK ──────────────────────────────────────────────────────────
    # Set MODEL_NAME in your .env to override, e.g.: MODEL_NAME=gemini-2.5-flash
    FALLBACK_MODEL = "gemini-2.5-flash"
    # ─────────────────────────────────────────────────────────────────────────

    api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
    if not api_key:
        raise EnvironmentError(
            "❌ T000: GEMINI_API_KEY not found. "
            "Add it to your .env file: GEMINI_API_KEY=your_key_here"
        )

    # ── Create Client (new SDK pattern) ───────────────────────────────────────
    # The Client object is the single point of truth for auth in google-genai.
    # It is passed down to T115 so no tile ever calls genai.configure() again.
    try:
        client = genai.Client(api_key=api_key)
        print("✅ T000: google-genai Client created.")
    except Exception as e:
        raise RuntimeError(f"❌ T000: Failed to create genai Client: {e}")

    # ── Model Discovery ────────────────────────────────────────────────────────
    available = []
    try:
        for m in client.models.list():
            name = getattr(m, 'name', '').replace('models/', '')
            # Filter to generative Gemini models only
            if 'gemini' in name.lower():
                available.append(name)
        available = sorted(list(set(available)))
        print(f"✅ T000: Discovered {len(available)} Gemini models.")
    except Exception as e:
        print(f"⚠️ T000: Model discovery warning: {e}. Using fallback list.")
        available = ["gemini-1.5-flash", "gemini-2.5-flash", "gemini-1.5-pro"]

    # ── Model Selection (env-driven, no blocking input()) ─────────────────────
    selected_model = env_loader.get("MODEL_NAME", FALLBACK_MODEL) or FALLBACK_MODEL

    if available and selected_model not in available:
        print(f"⚠️ T000: '{selected_model}' not in discovered list.")
        print(f"   Available models:")
        for m in available:
            print(f"     • {m}")
        print(f"   → Proceeding with '{selected_model}' (may still work if valid).")
    else:
        print(f"🚀 T000: Model locked → {selected_model}")

    # Return client so T115 reuses it — no double auth
    return "GEMINI", selected_model, client


In [2]:
# ==============================================================================================================================
# TILE T118: ENV_LOADER — VSCode Edition
# ==============================================================================================================================
# VERSION: 1.0.0 | STATUS: New
# PREDECESSORS: T000 (BIOS)
# INPUT: .env file on disk (same directory as notebook)
# PROCESSING: Loads environment variables via python-dotenv
# OUTPUT: EnvLoader instance — provides .get(key, fallback) interface
#         Drop-in replacement for google.colab.userdata.get()
# ------------------------------------------------------------------------------------------------------------------------------
# RATIONALE: google.colab.userdata is a Colab-exclusive API. T118 provides an identical
#             interface using python-dotenv so all upstream tiles (T000, Master Assembly)
#             require zero changes to their secret-fetching logic.
# DEPRECATION TRIGGER: If project migrates to a secret manager (e.g. HashiCorp Vault,
#                      AWS Secrets Manager), T118 evolves to T118b and this tile deprecates.
# ------------------------------------------------------------------------------------------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _DOTENV_AVAILABLE = True
except ImportError:
    _DOTENV_AVAILABLE = False
    print("⚠️ T118: python-dotenv not installed. Run: pip install python-dotenv")
    print("         Falling back to OS environment variables only.")

class EnvLoader:
    """
    VSCode-safe secrets interface.
    Replaces: google.colab.userdata
    Usage:    env_loader.get('GEMINI_API_KEY')
    """

    def __init__(self, env_path=".env"):
        self.env_path = Path(env_path).resolve()
        self.loaded = False

        if _DOTENV_AVAILABLE:
            if self.env_path.exists():
                load_dotenv(self.env_path, override=False)
                self.loaded = True
                print(f"✅ T118: ENV loaded → {self.env_path}")
            else:
                print(f"⚠️ T118: .env not found at {self.env_path}")
                print(f"         Create it with: GEMINI_API_KEY=your_key_here")
                print(f"         Falling back to OS environment variables.")
        else:
            print("⚠️ T118: Running on OS environment variables only (dotenv unavailable).")

    def get(self, key: str, fallback=None):
        """
        Fetch a secret by key. Mirrors google.colab.userdata.get() exactly.
        Priority: .env file → OS env var → fallback value
        """
        value = os.getenv(key, fallback)
        if value is None:
            print(f"⚠️ T118: Key '{key}' not found. Add it to your .env file.")
        return value

    def status(self):
        """Diagnostic: print all loaded keys (values masked for security)."""
        print(f"\n📋 T118 ENV STATUS — Source: {self.env_path}")
        print(f"   dotenv loaded: {self.loaded}")
        print(f"   dotenv available: {_DOTENV_AVAILABLE}")
        # List keys present in .env without revealing values
        if self.loaded and self.env_path.exists():
            with open(self.env_path) as f:
                keys = [line.split('=')[0].strip() for line in f
                        if line.strip() and not line.startswith('#') and '=' in line]
            for k in keys:
                masked = os.getenv(k, '<not loaded>')
                display_val = f"{masked[:4]}****{masked[-2:]}" if masked and len(masked) > 6 else "****"
                print(f"   ✓ {k} = {display_val}")


In [3]:
# ==========================================
# TILE T100: INIT
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolving
# ROLE: Conditional Environment Setup
# ------------------------------------------
import json, os, time, re
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

def initialize_env(provider):
    # VSCode: google-genai is imported at module level (pip install google-genai)
    # T000 already validated the Client — this just confirms the package is importable.
    if provider == "GEMINI":
        try:
            from google import genai  # new SDK: google-genai
            print("✅ T100: google-genai SDK verified.")
        except ImportError:
            raise ImportError("❌ T100: Run 'pip install google-genai' first.")
    else:
        # Placeholder for future OpenAI support (T116 evolution)
        print("✅ T100: OpenAI provider selected (T116 tile required).")

    # Shared Globals -----------------------
    globals()['stop_signal'] = False
    globals()['token_ledger'] = []
    globals()['current_scene_history'] = [] # Critical Fix
    return True

# ------------------------------------------

In [4]:
# ==========================================
# TILE T101: AI_GATEWAY
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolving
# ROLE: Multi-Provider Routing
# ------------------------------------------

class AIGateway:
    def __init__(self, provider, model_name, api_key):
        self.provider = provider
        if provider == "GEMINI":
            # Routing to T115
            self.engine = GeminiProvider(api_key, model_name)
        elif provider == "OPENAI":
            # Routing to T116 (to be forged)
            # self.engine = OpenAIProvider(api_key, model_name)
            pass

    def request(self, prompt):
        # The Brain (T106) calls this. Gateway handles the delegation.
        return self.engine.call(prompt)

# ------------------------------------------

In [5]:
# ==========================================
# TILE T102: QUOTA_SHIELD
# ==========================================
# VERSION: 5.3.0 | STATUS: Evolved
# PREDECESSORS: T115 (GeminiProvider)
# INPUT: Any callable + args (GeminiProvider.generate_content), optional log_fn
# PROCESSING: Executes call; on 429 sleeps cooldown then retries once;
#             routes ALL error messages through log_fn (not bare print)
# OUTPUT: (response_text: str | None, usage_metadata | None)
# ------------------------------------------
# VERSION CHANGE (5.2 → 5.3):
#   CRITICAL FIX: All error messages now routed through self.log_fn.
#   Previous version used bare print() which goes to kernel stdout —
#   invisible inside the Textarea console widget. This was the reason
#   API failures appeared as silent fallbacks with no visible error.
#   T111 passes its _log() method so errors appear in the console.
# ------------------------------------------

import time

class QuotaShield:
    def __init__(self, cooldown_seconds=65, log_fn=None):
        self.cooldown = cooldown_seconds
        # log_fn routes messages to wherever the UI wants them.
        # T111 sets this to self._log so errors appear in Textarea console.
        # Default: bare print (kernel stdout) as safe fallback.
        self.log_fn = log_fn if log_fn else print

    def set_log_fn(self, fn):
        """Called by T111 after construction to wire in the console logger."""
        self.log_fn = fn

    def _is_rate_limit(self, error):
        """Detect 429 / RESOURCE_EXHAUSTED across google-genai error formats."""
        err_str = str(error).lower()
        return (
            "429" in err_str or
            "resource_exhausted" in err_str or
            "rate limit" in err_str or
            "quota" in err_str or
            "too many requests" in err_str
        )

    def protect(self, func, *args, **kwargs):
        """
        Execute func. On 429: sleep cooldown then retry once.
        On other errors: fail fast, return (None, None).
        ALL messages routed through self.log_fn (visible in console Textarea).
        """
        for attempt in range(2):
            try:
                result = func(*args, **kwargs)

                if isinstance(result, tuple):
                    response_obj, usage = result
                else:
                    response_obj = result
                    usage = getattr(result, 'usage_metadata', None)

                if hasattr(response_obj, 'candidates'):
                    pass  # Valid envelope shape

                return response_obj.text, usage

            except Exception as e:
                if self._is_rate_limit(e):
                    if attempt == 0:
                        self.log_fn(f"⏳ T102: Rate limit hit. Cooling {self.cooldown}s before retry...")
                        time.sleep(self.cooldown)
                        self.log_fn("🔄 T102: Retrying after cooldown...")
                        continue
                    else:
                        self.log_fn(f"❌ T102: Rate limit persists after cooldown. Skipping turn.")
                        return None, None
                else:
                    self.log_fn(f"❌ T102 ERROR: {type(e).__name__}: {e}")
                    return None, None

        return None, None

# ------------------------------------------


In [6]:
# ==========================================
# TILE T103: IDENTITY_VAULT
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolving
# ROLE: Persona Construction & Mood Injection
# ARTEFACTS: noahs_ark_v3.json
# ------------------------------------------

class IdentityVault:
    def __init__(self, state_handle):
        self.state = state_handle

    def get_full_persona(self, actor_name, environment="Normal", p_tone="Neutral"):
        """
        Input: Actor Name, Environment, UI P-Tone
        Process: Merges Dossier + Mood + Tone + Env Logic
        Output: Formatted Persona String
        """
        data = self.state.get_actor_data(actor_name)

        # 1. Base Data Retrieval -------------------
        persona = data.get('identity', 'Unknown Entity')
        dossier = data.get('dossier', 'No records found.')
        mood = data.get('mood', 'Steady')

        # 2. Environmental Logic -------------------
        # Rationale: Dangerous environments trigger 'Alert' mood
        if "Dangerous" in environment or "Hostile" in environment:
            mood = "Alert/Defensive"

        # 3. Constructing the Identity Block -------
        full_string = f"""
        CORE IDENTITY: {persona}
        CURRENT MOOD: {mood}
        NARRATIVE TONE: {p_tone}
        STAGING ENVIRONMENT: {environment}
        DOSSIER: {dossier}
        """
        return full_string.strip()

# ------------------------------------------

In [7]:
# ==========================================
# TILE T104: CONTEXT_SLIDER
# ==========================================
# VERSION: 5.0.0 | STATUS: Evolving
# ROLE: Memory Truncation (The Snowball Fix)
# ARTEFACTS: mission_log List Buffer
# ------------------------------------------

class ContextSlider:
    def __init__(self, window_size=7):
        # Rationale: Miller's Law (7 +/- 2) for optimal LLM focus
        self.window_size = window_size

    def slide(self, history_list):
        """Input: Full Dialogue List | Process: Slicing | Output: Formatted Str"""
        if not history_list:
            return "No previous dialogue recorded."

        # Context Truncation -----------------------
        truncated_list = history_list[-self.window_size:]

        # Return as single prompt-ready string -----
        return "\n".join(truncated_list)

# ------------------------------------------

In [8]:
# ==========================================
# TILE T105: MAYA_META_OBSERVER
# ==========================================
# VERSION: 5.0.0 | STATUS: New
# ROLE: High-Level Narrator & Plot Facilitator
# ARTEFACTS: Narrative Bridge
# ------------------------------------------

class MayaMetaObserver:
    def __init__(self, model_handle, shield_handle):
        self.model = model_handle
        self.shield = shield_handle

    def narrate(self, environment, objective, history_list):
        """
        Input: Env, Goal, and Dialogue History
        Process: Generates a 'Meta-Commentary' or 'Narrative Bridge'
        Output: Text string for the Mission Log
        """
        prompt = f"""
        ROLE: You are 'Maya', the invisible narrator of this simulation.
        ENVIRONMENT: {environment}
        MISSION OBJECTIVE: {objective}

        RECENT EVENTS:
        {history_list[-3:] if history_list else "The scene begins."}

        INSTRUCTIONS:
        - Provide 1-2 sentences of atmospheric narration.
        - Do NOT speak for characters.
        - Describe environmental changes or the 'feeling' of the scene.
        """

        # Using T102 (Shield) for the call
        raw_text, usage = self.shield.protect(self.model.generate_content, prompt)
        return f"[MAYA]: {raw_text}" if raw_text else ""

# ------------------------------------------

In [9]:
# ==========================================
# TILE T106: CHARACTER_BRAIN
# ==========================================
# VERSION: 5.3.0 | STATUS: Evolved
# PREDECESSORS: T102 (Shield), T103 (Vault→persona), T104 (Slider→context)
# INPUT: persona string, context string, environment, objective
# PROCESSING: Assembles structured prompt; calls API via Shield;
#             extracts [Urge] value via regex; returns formatted turn
# OUTPUT: (formatted_text: str, urge_value: int, token_map: dict)
# ------------------------------------------
# VERSION CHANGE (5.2.8 → 5.3.0):
#   - CRITICAL FIX A: generate() prompt now requests explicit
#     [Thought]/[Urge]/[Speech] structure matching Chapter 2 format.
#     Previous prompt said "Keep it concise" with no format spec —
#     model produced minimal/empty output.
#   - CRITICAL FIX B: urge value now extracted via regex from response.
#     Previously always returned as string "None".
#   - generate_urge() (Oracle path) unchanged.
# ------------------------------------------

import re

class CharacterBrain:
    def __init__(self, model_handle, shield_handle, sentinel_handle, registry_handle):
        self.model    = model_handle    # T115: GeminiProvider
        self.shield   = shield_handle   # T102: QuotaShield
        self.sentinel = sentinel_handle # T108: TokenSentinel
        self.r        = registry_handle # Central registry

    # ── Oracle Path ───────────────────────────────────────────────────────────

    def generate_urge(self, actor_name, custom_prompt):
        """Oracle query path — direct prompt, no scene structure."""
        print(f"System: Oracle query for {actor_name}...")

        if "token" in custom_prompt.lower() or "usage" in custom_prompt.lower():
            ledger_data   = self.sentinel.get_ledger_summary()
            custom_prompt = f"SYSTEM DATA: {ledger_data}\n\nUSER QUERY: {custom_prompt}"

        raw_text, usage = self.shield.protect(self.model.generate_content, custom_prompt)
        tokens = {
            "p": getattr(usage, "prompt_token_count", 0),
            "c": getattr(usage, "candidates_token_count", 0)
        }
        self.sentinel.log(tokens, actor=actor_name)
        return raw_text, 5, tokens

    # ── Simulation Path ───────────────────────────────────────────────────────

    def _extract_urge(self, text: str) -> int:
        """
        Regex extraction of [Urge] value from structured response.
        Matches: [Urge]: 7  or  [Urge]: 7 (reason text)
        Returns int 1-10, defaults to 5 if not found.
        """
        if not text:
            return 5
        match = re.search(r'\[Urge\]:\s*(\d+)', text)
        if match:
            return min(10, max(1, int(match.group(1))))
        return 5

    def generate(self, persona, context, env, obj):
        """
        Main simulation turn generator.
        Returns: (formatted_text: str, urge: int, token_map: dict)
        Output format matches Chapter 2 reference:
            [Thought]: internal reasoning
            [Urge]: N (motivation description)
            [Speech]: spoken words or action
        """

        # ── Structured Prompt ─────────────────────────────────────────────────
        full_prompt = f"""You are an AI actor in an immersive simulation. Respond ONLY in character.

STAGING ENVIRONMENT:
{env}

SCENE OBJECTIVE:
{obj}

YOUR COMPLETE IDENTITY & DOSSIER:
{persona}

RECENT SCENE HISTORY (last exchanges):
{context if context and context != "No previous dialogue recorded." else "The scene is just beginning."}

─────────────────────────────────────────────────────────────────
RESPONSE FORMAT — follow this EXACTLY, no deviations:

[Thought]: <Your unspoken internal reasoning. What are you thinking, sensing, calculating? Be specific and personal to your character. 2-5 sentences.>

[Urge]: <Integer 1-10> (<One sentence describing what impulse or drive is guiding your next action. Higher = more intense.>)

[Speech]: <Your spoken words, actions, or physical reactions. Use italics (*like this*) for physical actions. Be authentic to your character voice. No length limit — be as rich as the moment demands.>
─────────────────────────────────────────────────────────────────

Rules:
- Stay fully in character. Never break the fourth wall.
- [Thought] is private — the reader sees it but other characters cannot.
- [Urge] must be a single integer followed by a brief motivation.
- [Speech] is what actually happens in the scene.
- Do not add any text before [Thought] or after [Speech].
"""

        # ── API Call via Shield ────────────────────────────────────────────────
        txt, usage = self.shield.protect(self.model.generate_content, full_prompt)

        # ── Token Map ─────────────────────────────────────────────────────────
        token_map = {
            "p": getattr(usage, "prompt_token_count", 0),
            "c": getattr(usage, "candidates_token_count", 0)
        }

        # ── Urge Extraction ───────────────────────────────────────────────────
        urge_value = self._extract_urge(txt)

        # ── Fallback if response is empty ─────────────────────────────────────
        if not txt or not txt.strip():
            txt = "[Thought]: The moment stretches in silence.\n[Urge]: 3 (To wait and observe.)\n[Speech]: *No response yet.*"

        return txt, urge_value, token_map


In [10]:
# ==========================================
# TILE T107: STATE_ENGINE
# ==========================================
# VERSION: 5.5.0 | STATUS: Evolved
# PREDECESSORS: None (Core State Layer)
# INPUT: Soul data (noah_souls.json), history events from T114
# PROCESSING: Manages all persistent world state — souls + global history
# OUTPUT: Participant data (→ T103 Vault), history (→ T104 Slider, T110 Oracle)
# ARTEFACTS: noah_souls.json, global_history.json
# ------------------------------------------
# VERSION CHANGE (5.4.0 → 5.5.0):
#   - CRITICAL FIX A: load_history() now filters '...' and 'No response yet'
#     entries so corrupted runs don't contaminate new scenes on restart.
#   - CRITICAL FIX B: Added reset_scene() — clears global_history in memory
#     and on disk. Called by T111 New Scene button before each simulation.
#     Souls are preserved; only narrative history is wiped.
# ------------------------------------------

import json
import os

class StateEngine:
    def __init__(self, souls_file="noah_souls.json", history_file="global_history.json"):
        self.souls_file   = souls_file
        self.history_file = history_file

        self.default_souls = {
            "Noah": {"identity": "The weary but visionary architect of Noah's Ark.", "mood": "Godly", "tone": "Wise", "dossier": "Creator and overseer of this world."},
            "Abdul Basha": {"identity": "33, US, Specialist in Extreme Flight Ops", "mood": "Neutral", "tone": "Professional", "dossier": "Expert aviator and crisis navigator."},
            "Rebecca Heut": {"identity": "27, UK, Specialist in Exobiology", "mood": "Neutral", "tone": "Inquisitive", "dossier": "Studies life in extreme environments."},
            "Vel Murugan": {"identity": "29, India, Specialist in Life-Support", "mood": "Neutral", "tone": "Anxious", "dossier": "Keeps the crew alive under pressure."},
            "Valentina Tereshkova": {"identity": "39, Russia, Specialist in Orbital Mechanics", "mood": "Neutral", "tone": "Calculated", "dossier": "Charts the course through space."},
            "Maya": {"identity": "Specialist AGI v2.4, Core of this world", "mood": "Analytical", "tone": "Enigmatic", "dossier": "The silent observer who sees all."}
        }

        self.state = {
            "global_history": [],
            "participants": {}
        }
        self.state['participants'] = self.load_souls()
        self.state['global_history'] = self.load_history()

    # ── Souls (Participants) ───────────────────────────────────────────────────

    def load_souls(self):
        if os.path.exists(self.souls_file):
            try:
                with open(self.souls_file, 'r') as f:
                    return json.load(f)
            except:
                return self.default_souls
        with open(self.souls_file, 'w') as f:
            json.dump(self.default_souls, f, indent=4)
        return self.default_souls

    def save_souls(self, souls_dict):
        with open(self.souls_file, 'w') as f:
            json.dump(souls_dict, f, indent=4)
        self.state['participants'] = souls_dict

    def get_actor_data(self, actor_name):
        """Bridge for IdentityVault — returns full soul record."""
        return self.state['participants'].get(actor_name, {})

    # ── Global History (Chronicles) ───────────────────────────────────────────

    def load_history(self):
        """Load persisted global history from disk on init."""
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list) and data:
                        # Filter out fallback/empty entries from corrupted runs
                        clean = [e for e in data if not e.endswith('...')
                                 and 'No response yet' not in e]
                        if clean:
                            print(f"✅ T107: Loaded {len(clean)} history entries "
                                  f"({len(data)-len(clean)} empty entries filtered).")
                            return clean
            except:
                pass
        return []

    def reset_scene(self):
        """
        CRITICAL: Wipe global_history for a fresh new scene run.
        Clears both in-memory state and the on-disk global_history.json.
        Souls (noah_souls.json) are preserved — only narrative history is cleared.
        Call this via the New Scene button in T111 before each new simulation.
        """
        self.state['global_history'] = []
        try:
            with open(self.history_file, 'w') as f:
                json.dump([], f)
            print(f"✅ T107: Scene reset. global_history.json cleared.")
        except Exception as e:
            print(f"⚠️ T107: Scene reset failed: {e}")

    def save_history(self):
        """Persist global history to disk. Called after every log_event()."""
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.state['global_history'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: History save failed: {e}")

    def update_global_history(self, entry):
        """Add a turn to chronicles and immediately persist to disk."""
        if 'global_history' not in self.state:
            self.state['global_history'] = []
        self.state['global_history'].append(entry)
        self.save_history()  # CRITICAL FIX: persist on every entry

    def log_event(self, entry):
        """Alias used by SimConductor (T114)."""
        self.update_global_history(entry)

# ------------------------------------------


In [11]:
# ==========================================
# TILE T108: TOKEN_SENTINEL
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolved
# ROLE: Economic Audit & Token Tracking
# ARTEFACTS: token_analytics.json
# ------------------------------------------
# Version Remarks: retains the "Atomic Append" logic that fixed your persistence issue but restores the get_ledger_summary method so the Oracle can report stats in chat
# ------------------------------------------

import json
from datetime import datetime
import os

class TokenSentinel:
    def __init__(self, storage_file="token_ledger.json"):
        self.storage_file = storage_file
        self.P_RATE = 0.00000125
        self.C_RATE = 0.00000375
        self.session_ledger = self._load_ledger()

    def _load_ledger(self):
        """Standard JSON load with safety check"""
        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, 'r') as f:
                    data = json.load(f)
                    return data if isinstance(data, list) else []
            except (json.JSONDecodeError, IOError):
                return []
        return []

    def _save_ledger(self, new_entry):
        """Atomic Append: Reload, append, then save."""
        current_disk_data = self._load_ledger()
        current_disk_data.append(new_entry)
        self.session_ledger = current_disk_data
        with open(self.storage_file, 'w') as f:
            json.dump(current_disk_data, f, indent=4)

    def audit_turn(self, actor, tokens_dict):
        p_count = tokens_dict.get('p', 0)
        c_count = tokens_dict.get('c', 0)

        # FIX v5.1.0: Skip zero-token entries — these are failed API calls
        # (rate-limited or errored). Logging them pollutes the ledger with
        # meaningless rows. T102 now handles retries so genuine turns always
        # produce real token counts.
        if p_count == 0 and c_count == 0:
            return sum(item['usd'] for item in self.session_ledger)

        cost = (p_count * self.P_RATE) + (c_count * self.C_RATE)
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "actor": actor,
            "prompt": p_count,
            "completion": c_count,
            "usd": cost
        }
        self._save_ledger(entry)
        return sum(item['usd'] for item in self.session_ledger)

    def log(self, tokens_dict, actor="Oracle"):
        return self.audit_turn(actor, tokens_dict)

    def get_totals(self):
        """Standardized for T110 Oracle RAG"""
        data = self._load_ledger()
        if not data:
            return {"p": 0, "c": 0, "usd": 0.0}
        p = sum(item.get('prompt', 0) for item in data)
        c = sum(item.get('completion', 0) for item in data)
        usd = sum(item.get('usd', 0.0) for item in data)
        return {"p": p, "c": c, "usd": usd}

    def get_ledger_summary(self):
        """ORACLE TRIGGER: RESTORED method for the Oracle's sight"""
        current_data = self._load_ledger()
        if not current_data: return "The ledger is empty."
        summary = "TOKEN USAGE REPORT:\n"
        for entry in current_data:
            summary += f"- {entry['timestamp']} | {entry['actor']}: {entry['prompt'] + entry['completion']} tokens (${entry['usd']:.6f})\n"
        return summary

In [12]:
# ==========================================
# TILE T109: LOG_ARCHIVER
# ==========================================
# VERSION: 5.0.0 | STATUS: Evolving
# ROLE: Data Archival & Exit Procedures
# ARTEFACTS: Archive_Scene_[Timestamp].txt
# ------------------------------------------

class LogArchiver:
    def archive(self, environment, objective, history_list):
        """Input: Mission Metadata | Process: Disk Write | Output: Filename"""
        # Timestamp Generation ---------------------
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"Archive_Scene_{timestamp}.txt"

        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write(f"--- NOAH'S ARK MISSION ARCHIVE ---\n")
                f.write(f"TIMESTAMP: {timestamp}\n")
                f.write(f"ENVIRONMENT: {environment}\n")
                f.write(f"OBJECTIVE: {objective}\n")
                f.write("-" * 40 + "\n\n")

                # Write History --------------------
                for turn in history_list:
                    f.write(f"{turn}\n")

            print(f"✅ T109: Archive created: {filename}")
            return filename
        except Exception as e:
            print(f"❌ T109 ERROR: Archival failure - {e}")
            return None

# ------------------------------------------

In [13]:
# ==========================================
# TILE T110: ORACLE_LOGIC
# ==========================================
# VERSION: 5.3.1 | STATUS: Full Library Access
# Version Remarks: Brought back Oracles ability to read registries and ledgers to answer
# ------------------------------------------
import json
import os

class OracleLogic:
    def __init__(self, registry):
        self.r = registry

    def ask(self, user_query):
        # 1. READ TOKEN LEDGER (Economics)  [show_thinking() removed — method not yet on UIOracleTab]
        token_data = self.r['sentinel'].get_totals()

        # 2. READ SOULS LIBRARY (Personnel)
        # We pull directly from the state engine which manages noah_souls.json
        souls = self.r['state'].state.get('participants', {})
        souls_list = ", ".join([f"{name} ({info.get('tone', 'Unknown')})" for name, info in souls.items()])

        # 3. READ GLOBAL HISTORY (Past Actions)
        history = self.r['state'].state.get('global_history', [])[-15:] # Last 15 events

        # 4. CONSTRUCT THE "ULTIMATE REALITY" CONTEXT
        # This gives the AI access to every major JSON-backed data point
        context = f"""
        --- ARK MANIFEST ---
        PERSONNEL ON BOARD: {souls_list}

        --- ECONOMIC LEDGER ---
        TOTAL USD: ${token_data['usd']:.6f}
        TOTAL TOKENS: {token_data['p'] + token_data['c']}

        --- RECENT CHRONICLES ---
        {history if history else "The chronicles are currently empty."}
        """

        system_instruction = (
            "You are the Oracle of Noah's Ark. You have 'Sovereign Sight' over all the scrolls of the wonderful Noah's Ark. "
            "Use the provided MANIFEST, LEDGER, and CHRONICLES to answer your creator Noah's questions. "
            "If asked about people, refer to the Manifest. If asked about money, refer to the Ledger. "
            "Respond with respect to Noah your creator, be poetic, wise, and factually grounded with only the Noah Ark's data."
        )

        full_prompt = f"{system_instruction}\n\n[SIGHT DATA]\n{context}\n\n[USER QUERY]\n{user_query}"

        # 5. EXECUTE & LOG via Shield (returns text_str, usage_meta)
        response_text, usage_meta = self.r['shield'].protect(
            self.r['gateway'].generate_content, full_prompt
        )
        if usage_meta:
            tokens = {
                'p': getattr(usage_meta, 'prompt_token_count', 0),
                'c': getattr(usage_meta, 'candidates_token_count', 0)
            }
            self.r['sentinel'].log(tokens, actor="Oracle")

        return response_text if response_text else "The Oracle is silent."  

In [14]:
# ==========================================
# TILE T111: COMMAND_CENTER
# ==========================================
# VERSION: 5.6.0 | STATUS: Evolved
# PREDECESSORS: T107 (State), T114 (Conductor), T109 (Archiver),
#               T102 (Shield — wired for log_fn), T115 (Provider — wired for log_fn)
# INPUT: User button clicks, Soul Forge entries, Cockpit config
# PROCESSING: Manages UI; wires T102/T115 log_fn after registry is available;
#             spawns simulation thread; routes turns via T114
# OUTPUT: Console display (Textarea), triggers archive and scene-reset
# ------------------------------------------
# VERSION CHANGE (5.5.0 → 5.6.0):
#   FIX A: wire_logging() routes T102+T115 error messages to Textarea console.
#           Previously errors went to kernel stdout — completely invisible.
#   FIX B: "🆕 New Scene" button added. Calls T107.reset_scene() before
#           starting. Without this, old history (including "..." corruption)
#           contaminates every new run.
#   FIX C: "🔬 Test API" button added. Single live call to verify the API key,
#           Client, and model are all working before running a full scene.
#   FIX D: _start_sim auto-calls reset_scene() so New Scene is enforced
#           every time ▶ Start is pressed. History never bleeds between scenes.
# ------------------------------------------

import ipywidgets as widgets
import time
import traceback
import threading
from IPython.display import display

class CommandCenter:
    def __init__(self, registry):
        self.r = registry
        self.is_running = False
        self.soul_rows = []
        self.active_config = {}

        # ── Soul Forge widgets ────────────────────────────────────────────────
        self.soul_canvas = widgets.VBox(
            layout={'height': '200px', 'overflow_y': 'scroll', 'border': '1px solid #444'})
        self.add_btn  = widgets.Button(description="➕ Add Soul",    button_style='info')
        self.save_btn = widgets.Button(description="💾 Save Souls",  button_style='success')
        self.soul_status = widgets.Label(value="Status: Click Add to create souls")

        # ── Cockpit widgets ───────────────────────────────────────────────────
        self.env_box = widgets.Textarea(
            placeholder="Environment Setup...",
            layout={'width': '98%', 'height': '60px'})
        self.obj_box = widgets.Textarea(
            placeholder="Scene Objective...",
            layout={'width': '98%', 'height': '60px'})

        self.cockpit_canvas = widgets.VBox(
            layout={'height': 'auto', 'min_height': '200px', 'overflow': 'visible'})
        self.cockpit_scroll_container = widgets.Box(
            [self.cockpit_canvas],
            layout={'height': '250px', 'overflow_y': 'scroll', 'border': '1px solid #444'})

        # ── Control buttons ───────────────────────────────────────────────────
        self.new_scene_btn = widgets.Button(
            description="🆕 New Scene",
            button_style='warning',
            tooltip="Clears history and resets the world for a fresh scene"
        )
        self.test_btn = widgets.Button(
            description="🔬 Test API",
            button_style='info',
            tooltip="Send a single test call to verify API key and model are working"
        )
        self.start_btn = widgets.Button(description="▶ Start",     button_style='success')
        self.pause_btn = widgets.Button(description="⏸ Pause",     button_style='warning')
        self.kill_btn  = widgets.Button(description="⏹ Hard Kill", button_style='danger')

        # ── Console: Textarea for reliable VSCode scroll ──────────────────────
        self.console = widgets.Textarea(
            value='',
            disabled=True,
            layout=widgets.Layout(
                height='300px',
                width='100%',
                overflow_y='scroll',
                border='1px solid #555',
                font_family='monospace',
                font_size='12px'
            )
        )

        # ── Tab assembly ──────────────────────────────────────────────────────
        self.sub_tabs = widgets.Tab()
        self.sub_tabs.children = [self._ui_soul_forge(), self._ui_cockpit()]
        self.sub_tabs.set_title(0, "✨ Souls")
        self.sub_tabs.set_title(1, "🚀 Cockpit")
        self.sub_tabs.observe(self._on_sub_tab_change, names='selected_index')

        self._refresh_cockpit()

    def wire_logging(self):
        """
        FIX A: Route T102 and T115 error messages into the Textarea console.
        Must be called AFTER registry is populated (in Master Assembly, after
        all tiles are instantiated). Without this, T102/T115 errors print to
        kernel stdout (below the cell) — completely invisible to the user.
        """
        if 'shield' in self.r:
            self.r['shield'].set_log_fn(self._log)
        if 'gateway' in self.r:
            self.r['gateway'].set_log_fn(self._log)
        self._log("🔌 T111: Console logging wired into T102 + T115.")

    # ── Soul Forge ────────────────────────────────────────────────────────────

    def _ui_soul_forge(self):
        self.add_btn.on_click(self._add_row)
        self.save_btn.on_click(self._save_souls)
        return widgets.VBox([
            self.soul_canvas,
            widgets.HBox([self.add_btn, self.save_btn]),
            self.soul_status
        ])

    def _add_row(self, _):
        row = widgets.HBox([
            widgets.Text(placeholder="Name"),
            widgets.Text(placeholder="Identity"),
            widgets.Text(placeholder="Tone")
        ])
        self.soul_rows.append(row)
        self.soul_canvas.children = list(self.soul_canvas.children) + [row]

    def _save_souls(self, _):
        souls = self.r['state'].load_souls()
        for row in self.soul_rows:
            name, ident, tone = [c.value for c in row.children]
            if name:
                souls[name] = {
                    "identity": ident, "mood": "Neutral",
                    "tone": tone, "dossier": ident
                }
        self.r['state'].save_souls(souls)
        self.soul_rows = []
        self.soul_canvas.children = []
        self.soul_status.value = "Status: Souls born into Noah's Ark."
        self._refresh_cockpit()

    # ── Cockpit ───────────────────────────────────────────────────────────────

    def _ui_cockpit(self):
        self.start_btn.on_click(self._start_sim)
        self.kill_btn.on_click(self._kill_sim)
        self.new_scene_btn.on_click(self._new_scene)
        self.test_btn.on_click(self._test_api)
        return widgets.VBox([
            widgets.HBox([
                widgets.VBox([
                    widgets.Label("🌍 Environment:"), self.env_box,
                    widgets.Label("🎯 Objective:"),   self.obj_box
                ], layout={'width': '40%'}),
                widgets.VBox([
                    widgets.Label("👥 Mission Personnel:"),
                    self.cockpit_scroll_container
                ], layout={'width': '60%'})
            ]),
            widgets.HBox([
                self.new_scene_btn, self.test_btn,
                self.start_btn, self.pause_btn, self.kill_btn
            ]),
            self.console
        ])

    def _on_sub_tab_change(self, change):
        if change['new'] == 1:
            self._refresh_cockpit()

    def _refresh_cockpit(self):
        souls = self.r['state'].load_souls()
        header = widgets.HBox([
            widgets.Label("Character Name",       layout={'width': '160px'}),
            widgets.Label("Involvement / Status", layout={'width': '160px', 'margin': '0 0 0 10px'})
        ], layout={'border': '1px solid #555'})

        rows = [header]
        self.active_config = {}
        for name in souls:
            dd = widgets.Dropdown(
                options=['Active', 'Passive', 'No'], value='No',
                layout={'width': '150px'}, style={'description_width': '0px'}
            )
            row = widgets.HBox(
                [widgets.Label(name, layout={'width': '160px'}), dd],
                layout={'padding': '2px 0'}
            )
            self.active_config[name] = dd
            rows.append(row)
        self.cockpit_canvas.children = rows

    def get_render_box(self):
        return self.sub_tabs

    # ── Logging ───────────────────────────────────────────────────────────────

    def _log(self, message):
        """
        Thread-safe console writer. Textarea is safe from daemon threads.
        MAX_LINES cap prevents unbounded memory in long sessions.
        """
        MAX_LINES = 400
        current = self.console.value
        lines   = current.split("\n") if current else []
        lines.append(str(message))
        if len(lines) > MAX_LINES:
            lines = lines[-MAX_LINES:]
        self.console.value = "\n".join(lines)

    # ── New Scene ─────────────────────────────────────────────────────────────

    def _new_scene(self, _):
        """
        FIX B: Reset all narrative history for a fresh scene.
        Clears global_history.json (T107) and wipes console.
        Souls are preserved.
        """
        if self.is_running:
            self._log("⚠️ Stop the simulation before starting a new scene.")
            return
        self.r['state'].reset_scene()
        self.console.value = ""
        self._log("🆕 New Scene started. History cleared. Souls preserved.")
        self._log("   Set your Environment and Objective, then press ▶ Start.")

    # ── API Test ──────────────────────────────────────────────────────────────

    def _test_api(self, _):
        """
        FIX C: Single live API call to verify the full T118→T000→T115→T102 chain.
        Result shown in console textarea — no guessing where the error is.
        """
        self._log("\n🔬 API TEST: Sending single test call...")

        def run_test():
            try:
                test_prompt = (
                    "Reply with exactly one sentence confirming you are online "
                    "and what model you are."
                )
                txt, usage = self.r['shield'].protect(
                    self.r['gateway'].generate_content, test_prompt
                )
                if txt and txt.strip():
                    p = getattr(usage, 'prompt_token_count', 0)
                    c = getattr(usage, 'candidates_token_count', 0)
                    self._log(f"✅ API TEST PASSED")
                    self._log(f"   Response: {txt.strip()[:150]}")
                    self._log(f"   Tokens — Prompt: {p} | Completion: {c}")
                else:
                    self._log("❌ API TEST: Call returned empty response.")
                    self._log("   Check T102/T115 errors above for details.")
            except Exception as e:
                self._log(f"❌ API TEST EXCEPTION: {type(e).__name__}: {e}")

        threading.Thread(target=run_test, daemon=True).start()

    # ── Simulation Engine ─────────────────────────────────────────────────────

    def _run_simulation_loop(self):
        """Main simulation loop — daemon thread."""
        try:
            while self.is_running:
                active_souls = [
                    name for name, dd in self.active_config.items()
                    if dd.value == 'Active'
                ]

                if not active_souls:
                    self._log("⚠️ No Active souls selected. Standing by...")
                    time.sleep(3)
                    continue

                for soul in active_souls:
                    if not self.is_running:
                        break
                    self._log(f"\n🌀 {soul} is formulating a response...")
                    try:
                        turn_text = self.r['conductor'].run_turn(soul)
                        self._log(f"💬 {turn_text}")
                    except Exception as e:
                        self._log(f"❌ ERROR in {soul}'s turn: {e}")
                        self._log(traceback.format_exc())
                        self.is_running = False
                        break
                    time.sleep(2)

        except Exception as fatal_e:
            self._log(f"💥 FATAL THREAD CRASH: {fatal_e}")
            self._log(traceback.format_exc())
            self.is_running = False

    def _start_sim(self, _):
        """
        FIX D: Automatically resets scene history before every run.
        Ensures no bleed from previous scenes or corrupted runs.
        """
        if not self.is_running:
            # Auto-reset scene so history never bleeds between runs
            self.r['state'].reset_scene()
            self.is_running = True
            self.console.value = ""
            self._log("🚀 Noah's Ark OS: Scene Initialized.")
            self._log(f"🎯 Objective: {self.obj_box.value[:120]}")
            self._log("-" * 50)
            threading.Thread(target=self._run_simulation_loop, daemon=True).start()

    def _kill_sim(self, _):
        """Hard Kill: stop engine, archive conversation, save history."""
        self.is_running = False
        self._log("⏹ Hard Kill signal sent. Saving world state...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])

            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")

            if 'archiver' in self.r:
                archive_file = self.r['archiver'].archive(env, obj, history)
                if archive_file:
                    self._log(f"📦 Scene archived → {archive_file}")
            else:
                self._log("⚠️ Archiver not in registry — skipping archive.")

            stats = self.r['sentinel'].get_totals()
            self._log(f"\n📊 Session Summary:")
            self._log(f"   Cost:   ${stats['usd']:.6f}")
            self._log(f"   Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("\n✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Save error during kill: {e}")

# ------------------------------------------


In [15]:
# ==========================================
# TILE T112: UI_ORACLE_TAB
# ==========================================
# VERSION: 5.1.4 | STATUS: Clean Output
# ROLE: Oracle Gateway with User Query Input
# ------------------------------------------
# Version Remarks: to update the on_submit logic in your Oracle tile to "unpack" that package and only display the first item (the text),so extra characters dont appear in Oracle response
# ------------------------------------------

import ipywidgets as widgets

class UIOracleTab:
    def __init__(self):
        self.r = None # Registry link
        self.output = widgets.Output()
        self.input_text = widgets.Text(placeholder="Ask your Oracle...")
        self.submit_btn = widgets.Button(description="Consult Oracle", button_style='info')

    def on_submit(self, _):
        query = self.input_text.value
        if not query: return

        with self.output:
            # 1. Clear input
            self.input_text.value = ""

            # 2. Route through OracleLogic (T110) — gives full contextual sight
            # oracle is stored in registry by Master Assembly
            oracle_logic = self.r.get('oracle')
            if oracle_logic:
                oracle_text = oracle_logic.ask(query)
            else:
                # Fallback: direct brain call if oracle not in registry
                response_package = self.r['brain'].generate_urge("Oracle", query)
                oracle_text = response_package[0]

            # 3. Display response
            print(f"> User: {query}")
            print(f"👁️ Oracle: {oracle_text}")

            # 4. Footer
            stats = self.r['sentinel'].get_totals()
            print(f"\nSession Cost (USD): ${stats['usd']:.6f}")
            print(f"Tokens: P: {stats['p']} | C: {stats['c']}")
            print("-" * 30)

    def get_render_box(self):
        self.submit_btn.on_click(self.on_submit)
        return widgets.VBox([self.output, widgets.HBox([self.input_text, self.submit_btn])])

In [16]:
# ==========================================
# TILE T113: UI_METRICS
# ==========================================
# VERSION: 5.1.2 | STATUS: Purified & Reactive
# ROLE: Real-Time Financial Telemetry
# ARTEFACTS: Dashboard Labels
# ------------------------------------------
# Version Remarks: Match the naming expected by UI Chassis.It includes the refresh_data logic and the correctly named get_render_box method
# ------------------------------------------

import ipywidgets as widgets
from IPython.display import display

class UIMetrics:
    def __init__(self):
        # UI Element Instantiation
        self.cost_label = widgets.Label(value="Session Cost (USD): $0.000000")
        self.token_info = widgets.HTML(value="<b>Tokens:</b> P: 0 | C: 0")

        # Bridge Handle (Set during Master Assembly)
        self.sentinel_handle = None

    def refresh_data(self):
        """
        The On-Focus Trigger:
        Pulls fresh data from T108's ledger before rendering.
        """
        if self.sentinel_handle:
            # Accessing T108: TOKEN_SENTINEL totals
            stats = self.sentinel_handle.get_totals()

            # Dynamic Widget Update
            self.cost_label.value = f"Session Cost (USD): ${stats['usd']:.6f}"
            self.token_info.value = f"<b>Tokens:</b> Prompt: {stats['p']} | Completion: {stats['c']}"

    def get_render_box(self):
        """
        Standardized method for T117 (UI Chassis) to build the tab.
        """
        return widgets.VBox([self.cost_label, self.token_info])

    def display_metrics(self):
        """Manual Fallback Display"""
        display(self.get_render_box())

In [17]:
# ==========================================
# TILE T114: SIM_CONDUCTOR
# ==========================================
# VERSION: 5.1.3 | STATUS: Evolving
# ROLE: Orchestration & Turn-Queue Management
# ARTEFACTS: global_history.json
# VERSION REMARKS: redundancy removed - self.r.get('ui_cmd') - self.r['ui_cmd']
# ------------------------------------------

class SimConductor:
    def __init__(self, registry):
        self.r = registry

    def run_turn(self, actor):
        """Conducts a single turn for a specific actor."""
        # 1. Gather Context Safely
        ui = self.r.get('ui_cmd')
        if not ui:
            raise Exception("Registry Error: 'ui_cmd' not found. Check Master Assembly order.")

        env = ui.env_box.value
        obj = ui.obj_box.value

        # 2. Get History from State
        # We use .state.get to prevent crashes if the history hasn't started yet
        history = self.r['state'].state.get('global_history', [])
        context = self.r['slider'].slide(history)

        # 3. Get Persona from VAULT
        # Verified: Vault handles identity logic
        persona = self.r['vault'].get_full_persona(actor, environment=env)

        # 4. Brain Generation
        response, urge, tokens = self.r['brain'].generate(persona, context, env, obj)

        # 5. Audit & Token Tracking
        self.r['sentinel'].audit_turn(actor, tokens)

        # 6. Persistent Memory
        turn_text = f"{actor}: {response}"

        # SAFETY CHECK: Route to whichever method name you used in T107
        if hasattr(self.r['state'], 'log_event'):
            self.r['state'].log_event(turn_text)
        else:
            self.r['state'].update_global_history(turn_text)

        return turn_text

In [18]:
# ==============================================================================================================================
# TILE T115: GEMINI_PROV — google-genai Edition
# ==============================================================================================================================
# VERSION: 2.2.0 | STATUS: Evolved
# PREDECESSORS: T000 (BIOS — provides Client), T102 (QuotaShield calls generate_content)
# INPUT: genai.Client object (from T000), model_name string, prompt string
# PROCESSING: Calls client.models.generate_content(); safely pre-extracts text
#             before returning so T102 never encounters a raising .text property
# OUTPUT: (response_envelope, usage_metadata) — consumed by T102 QuotaShield
# ------------------------------------------------------------------------------------------------------------------------------
# VERSION CHANGE (2.0 → 2.1):
#   - CRITICAL FIX: response.text in google-genai SDK raises ValueError when
#     response contains non-text parts (inline_data, function_call, etc).
#     This exception was bubbling up through T102, which caught it as a generic
#     error and returned None,None — causing T106's "..." fallback on every call.
#   - Fix: _safe_extract_text() pre-extracts text inside T115, wrapping the
#     result as a plain string attribute that NEVER raises.
#   - Fallback chain: response.text → candidates parts → empty string.
#   - All real token counts are now preserved because usage_metadata is always
#     captured from the raw response before the text extraction attempt.
# INSTALL: pip install google-genai
# DEPRECATION TRIGGER: Migration to Claude/OpenAI → T115b/T116 evolution.
# ------------------------------------------------------------------------------------------------------------------------------

from google import genai as _genai_module

class GeminiProvider:
    """
    Wraps google-genai Client for standardised I/O consumed by QuotaShield (T102).
    Interface contract: generate_content(prompt: str) → (envelope, usage_metadata)
    envelope.text is ALWAYS a str (never None, never a raising property).
    """

    def __init__(self, api_key: str, model_name: str, client=None, log_fn=None):
        self.model_id = model_name.replace("models/", "")
        self.log_fn   = log_fn if log_fn else print  # routed after T111 sets it
        if client is not None:
            self.client = client
            print(f"✅ T115: Reusing T000 Client → {self.model_id}")
        else:
            self.client = _genai_module.Client(api_key=api_key)
            print(f"✅ T115: New Client created → {self.model_id}")

    def set_log_fn(self, fn):
        """Called by T111 after construction to wire in the console logger."""
        self.log_fn = fn

    def _safe_extract_text(self, response) -> str:
        """
        Safely extract text from a google-genai response.
        google-genai's response.text raises ValueError when non-text parts exist.
        This method never raises — it always returns a string (possibly empty).

        Extraction order:
          1. response.text (fast path — works for plain text responses)
          2. Walk candidates[0].content.parts collecting all text parts
          3. Return empty string as last resort (logged as warning)
        """
        # ── Path 1: Fast path ─────────────────────────────────────────────────
        try:
            text = response.text
            if text:
                return text
        except (ValueError, AttributeError):
            pass  # Non-text parts present — fall through to manual extraction

        # ── Path 2: Walk candidates manually ──────────────────────────────────
        try:
            parts = response.candidates[0].content.parts
            collected = []
            for part in parts:
                part_text = getattr(part, "text", None)
                if part_text:
                    collected.append(part_text)
            if collected:
                return "\n".join(collected)
        except (IndexError, AttributeError, TypeError):
            pass

        # ── Path 3: Last resort ───────────────────────────────────────────────
        self.log_fn("⚠️ T115: Could not extract text from response. Check candidates.")
        return ""

    def generate_content(self, prompt: str):
        """
        Standardised call consumed by QuotaShield (T102).
        Returns: (response_envelope, usage_metadata)
        response_envelope.text is ALWAYS a str — never raises.
        """
        try:
            raw_response = self.client.models.generate_content(
                model=self.model_id,
                contents=prompt
            )

            # ── Capture usage FIRST (before any text extraction) ──────────────
            usage = getattr(raw_response, "usage_metadata",
                            type("U", (), {"prompt_token_count": 0,
                                          "candidates_token_count": 0})())

            # ── Safe text extraction (NEVER raises) ────────────────────────────
            safe_text = self._safe_extract_text(raw_response)

            # ── Build guaranteed envelope ──────────────────────────────────────
            class Envelope: pass
            env = Envelope()
            env.text            = safe_text          # plain str, never raises
            env.candidates      = getattr(raw_response, "candidates", [])
            env.usage_metadata  = usage
            env._raw            = raw_response       # preserve for debugging

            return env, usage

        except Exception as e:
            self.log_fn(f"❌ T115 Call Error: {type(e).__name__}: {e}")
            class FailShield: pass
            f = FailShield()
            f.text       = ""
            f.candidates = []
            u = type("U", (), {"prompt_token_count": 0, "candidates_token_count": 0})()
            return f, u


In [19]:
# ==============================================================================================================================
# TILE T116: SMOKE_TEST
# ==============================================================================================================================
# VERSION: 1.2.2 | STATUS: Stable
# ROLE: End-to-End System Integrity Validation
# ------------------------------------------------------------------------------------------------------------------------------
# Version Remarks: Performs a Live Handshake—it sends a tiny request to Google to make sure the v1 endpoint is open.
# ------------------------------------------------------------------------------------------------------------------------------

class SmokeTester:
    def __init__(self, registry):
        self.r = registry

    def run_bft(self):
        print("🧪 T116: INITIALIZING FULL SYSTEM BFT...")
        results = {
            "Registry Check": self._test_registry(),
            "State Write": self._test_state(),
            "API Handshake": self._test_api()
        }

        print("\n--- BFT REPORT ---")
        all_passed = True
        for test, passed in results.items():
            status = "✅" if passed else "❌"
            print(f"{status} {test}: {'PASSED' if passed else 'FAILED'}")
            if not passed: all_passed = False

        if not all_passed:
            print("\n🚨 CRITICAL: BFT FAILED. DO NOT PROCEED TO UI.")
        return all_passed

    def _test_registry(self):
        # Checks if core organs are connected
        keys = ['state', 'brain', 'gateway', 'shield']
        return all(k in self.r for k in keys)

    def _test_state(self):
        # Checks if we can save/load data
        try:
            # Use the correct StateEngine method (save_souls, not save_state)
            souls = self.r['state'].load_souls()
            self.r['state'].save_souls(souls)  # round-trip: load then save back
            return True
        except: return False

    def _test_api(self):
        # THE CRITICAL STEP: Live Ping
        try:
            # We use the brain to generate a 1-word urge
            # This forces T115 to hit the Google server
            resp = self.r['brain'].generate_urge("System", "Ping")
            return len(resp) > 0
        except Exception as e:
            print(f"   [API ERROR]: {e}")
            return False

In [20]:
# ==========================================
# TILE T117: UI_CHASSIS
# ==========================================
# VERSION: 5.1.3 | STATUS: Evolving
# ROLE: Master Layout & Tab Management
# ------------------------------------------
# Version Remarks: to ensure T117 explicitly shares its internal tab controller so the Master Assembly can watch it.
# ------------------------------------------

import ipywidgets as widgets
from IPython.display import display

class UIChassis:
    def __init__(self, sov_tab, oracle_tab, metrics_tab):
        self.sov = sov_tab
        self.oracle = oracle_tab
        self.metrics = metrics_tab
        self.tab = None # This will hold the widget

    def assemble(self):
        # Creating the physical tab widget
        self.tab = widgets.Tab()
        self.tab.children = [
            self.sov.get_render_box(),
            self.oracle.get_render_box(),
            self.metrics.get_render_box()
        ]

        # Setting the titles exactly as they appear in your notebook
        self.tab.set_title(0, '🎮 Command Center')
        self.tab.set_title(1, '👁️ Oracle')
        self.tab.set_title(2, '📊 Metrics')

    def display(self):
        if self.tab:
            display(self.tab)

    def render(self):
        """Used by smart_display to refresh views"""
        pass

In [21]:
# ==========================================
# NOAH'S ARK v5.8.0: MASTER ASSEMBLY — Visible Errors Edition
# ==========================================
# Version Remarks: strict Instantiate → Register → Link order
# VSCode Change: T118 EnvLoader replaces google.colab.userdata.
# genai Change: T000+T115 migrated to google-genai (google.genai) Client pattern.
#-------------------------------------------


# ── Boot & Global Init (VSCode Edition) ──────────────────────────────────────
# T118: Load environment secrets from .env file
env_loader = EnvLoader()                          # T118: reads .env

# T000: BIOS — validate API + select model
provider, model_name, genai_client = boot_sequence(env_loader)  # T000 — now returns client too

# Extract key for direct use by GeminiProvider (fallback if client injection not used)
api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
# ─────────────────────────────────────────────────────────────────────────────

# Logic Engines (T107, T101, T102, T103)
state    = StateEngine()      # Noah Souls support
shield   = QuotaShield()      #
gateway  = GeminiProvider(api_key, model_name, client=genai_client) # T115: reuses T000 Client
vault    = IdentityVault(state) # RESTORED: T103

# Processing Tiles (T108, T104, T106, T105)
sentinel = TokenSentinel()    # Re-synced Name: T108
archiver = LogArchiver()      # T109: wired for Hard Kill archive
slider   = ContextSlider()    #
maya     = MayaMetaObserver(gateway, shield) # RESTORED: T105

# Create Empty Registry First (Solving the NameError)
registry = {}

# Instantiate Brain and Conductor with Registry
brain    = CharacterBrain(gateway, shield, sentinel, registry) #
conductor = SimConductor(registry) #


# Populate Registry (Central Nervous System)
registry.update({
    'state': state,
    'shield': shield,
    'gateway': gateway,
    'vault': vault,
    'sentinel': sentinel,
    'slider': slider,
    'brain': brain,
    'maya': maya,
    'conductor': conductor,
    'archiver': archiver      # T109: needed by T111 Hard Kill
})


# UI Components (T111, T112, T113)
ui_cmd    = CommandCenter(registry) # Tabbed UI
ui_oracle = UIOracleTab()
ui_met    = UIMetrics()

# Logic Components
registry['ui_cmd']    = ui_cmd
registry['conductor'] = conductor   # new

# 6. Instantiating and Linking UI to Logic
# FIX: Store oracle in registry so T112 UIOracleTab can find it via self.r['oracle']
oracle_logic       = OracleLogic(registry)
registry['oracle'] = oracle_logic
ui_oracle.logic    = oracle_logic   # also keep direct handle for compatibility
ui_oracle.r        = registry       #
ui_met.sentinel_handle = sentinel #

# 7. Final Assembly & Display (T117)
chassis = UIChassis(ui_cmd, ui_oracle, ui_met)
chassis.assemble() #

# 8. Global Observers
def system_observer(change):
    if change['new'] == 2: # Metrics Tab
        ui_met.refresh_data()
    elif change['new'] == 0: # Command Center Tab
        if hasattr(ui_cmd, '_refresh_cockpit'):
            ui_cmd._refresh_cockpit()

chassis.tab.observe(system_observer, names='selected_index')

# 8b. Wire T102 + T115 error logging → console Textarea (CRITICAL)
# Must be called AFTER ui_cmd exists and registry is fully populated.
ui_cmd.wire_logging()

# 🚀 LAUNCH
chassis.display()

✅ T118: ENV loaded → C:\SK\noahs-ark\.env
--- NOAH'S ARK OS v5.5: google-genai BOOT ---
✅ T000: google-genai Client created.
✅ T000: Discovered 28 Gemini models.
🚀 T000: Model locked → gemini-2.5-flash
✅ T115: Reusing T000 Client → gemini-2.5-flash
